# Yelp Spark - Transformation Phase

24 business questions (6 per team member).  
Each section:
1. Calls the question function → prints `explain()` plan  
2. Shows the first rows with `show()`  
3. Saves the full result to `results/<transformation>/` as CSV  

**Run from the project root** so that `src` is importable.

## Setup - Spark session & datasets

In [ ]:
import sys
from pathlib import Path

# Make sure project root is on sys.path when running the notebook directly
project_root = Path("__file__").resolve().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.spark.session import create_spark_session
from src.spark.load_data import load_dataset
from src.constants import RESULTS_DIR
from src.analysis.transformations.utils import save_csv


spark = create_spark_session()
print("Spark session ready.")

In [ ]:
business = load_dataset(spark, "business").cache()
review   = load_dataset(spark, "review")
user     = load_dataset(spark, "user")
checkin  = load_dataset(spark, "checkin").cache()
tip      = load_dataset(spark, "tip").cache()
photo    = load_dataset(spark, "photo").cache()

results_root = Path(RESULTS_DIR)
results_root.mkdir(parents=True, exist_ok=True)
print(f"Results will be written to: {results_root}")

## Business Analytics

In [ ]:
import src.analysis.transformations.business as t1

### Q1 - Top 10 cities by number of open businesses
**Operations:** `filter`, `groupBy`

In [ ]:
df_t1_q1 = t1.q1_top_cities_by_open_businesses(business)
df_t1_q1.show(truncate=False)
save_csv(df_t1_q1, results_root / "business", "q1_top_cities_by_open_businesses")

### Q2 - Top 5 businesses by review count per state (open only)
**Operations:** `filter`, `window (rank)`

In [ ]:
df_t1_q2 = t1.q2_top_businesses_per_state(business)
df_t1_q2.show(20, truncate=False)
save_csv(df_t1_q2, results_root / "business", "q2_top_businesses_per_state")

### Q3 - Average review stars per state
**Operations:** `join`, `groupBy`

In [ ]:
df_t1_q3 = t1.q3_avg_review_stars_per_state(business, review)
df_t1_q3.show(20, truncate=False)
save_csv(df_t1_q3, results_root / "business", "q3_avg_review_stars_per_state")

### Q4 - Top businesses in Pennsylvania with ≥ 100 reviews
**Operations:** `filter`, `join`, `groupBy`

In [ ]:
df_t1_q4 = t1.q4_top_businesses_in_pennsylvania(business, review)
df_t1_q4.show(truncate=False)
save_csv(df_t1_q4, results_root / "business", "q4_top_businesses_in_pennsylvania")

### Q5 - Business stars vs. city average (delta)
**Operations:** `window (avg)`

In [ ]:
df_t1_q5 = t1.q5_business_vs_city_avg_stars(business)
df_t1_q5.show(truncate=False)
save_csv(df_t1_q5, results_root / "business", "q5_business_vs_city_avg_stars")

### Q6 - Open restaurants rated ≥ 4.5 with ≥ 100 reviews
**Operations:** `filter`

In [ ]:
df_t1_q6 = t1.q6_top_open_restaurants(business)
df_t1_q6.show(truncate=False)
save_csv(df_t1_q6, results_root / "business", "q6_top_open_restaurants")

## User Analytics

In [ ]:
import src.analysis.transformations.user as t2

### Q1 - Elite users with ≥ 100 fans and ≥ 500 reviews
**Operations:** `filter`

In [ ]:
df_t2_q1 = t2.q1_top_elite_users(user)
df_t2_q1.show(truncate=False)
save_csv(df_t2_q1, results_root / "user", "q1_top_elite_users")

### Q2 - User activity stats grouped by join year
**Operations:** `groupBy`

In [ ]:
df_t2_q2 = t2.q2_user_stats_by_join_year(user)
df_t2_q2.show(truncate=False)
save_csv(df_t2_q2, results_root / "user", "q2_user_stats_by_join_year")

### Q3 - Top 5 users by useful votes per join-year cohort
**Operations:** `window (rank)`

In [ ]:
df_t2_q3 = t2.q3_top_users_by_useful_per_year(user)
df_t2_q3.show(30, truncate=False)
save_csv(df_t2_q3, results_root / "user", "q3_top_users_by_useful_per_year")

### Q4 - Top 20 most active tip writers
**Operations:** `groupBy`, `join`

In [ ]:
df_t2_q4 = t2.q4_top_tip_writers(user, tip)
df_t2_q4.show(truncate=False)
save_csv(df_t2_q4, results_root / "user", "q4_top_tip_writers")

### Q5 - Useful reviews (stars ≥ 4, useful ≥ 5) by elite users
**Operations:** `filter`, `join`

In [ ]:
df_t2_q5 = t2.q5_useful_reviews_by_elite_users(user, review)
df_t2_q5.show(truncate=False)
save_csv(df_t2_q5, results_root / "user", "q5_useful_reviews_by_elite_users")

### Q6 - Percentile rank of users by total compliments
**Operations:** `window (percent_rank)`

In [ ]:
df_t2_q6 = t2.q6_user_compliment_percentile(user)
df_t2_q6.show(truncate=False)
save_csv(df_t2_q6, results_root / "user", "q6_user_compliment_percentile")

## Review Analytics

In [ ]:
import src.analysis.transformations.review as t3

### Q1 - Highly-voted reviews (votes > 10, stars ≥ 4)
**Operations:** `filter`

In [ ]:
df_t3_q1 = t3.q1_highly_voted_reviews(review)
df_t3_q1.show(truncate=False)
save_csv(df_t3_q1, results_root / "review", "q1_highly_voted_reviews")

### Q2 - Review volume and average stars by year-month
**Operations:** `groupBy`

In [ ]:
df_t3_q2 = t3.q2_reviews_by_month(review)
df_t3_q2.show(30, truncate=False)
save_csv(df_t3_q2, results_root / "review", "q2_reviews_by_month")

### Q3 - Most recent review per business
**Operations:** `window (row_number)`

In [ ]:
df_t3_q3 = t3.q3_latest_review_per_business(review)
df_t3_q3.show(truncate=False)
save_csv(df_t3_q3, results_root / "review", "q3_latest_review_per_business")

### Q4 - Top 20 businesses by total reviews and average score
**Operations:** `groupBy`, `join`

In [ ]:
df_t3_q4 = t3.q4_top_businesses_by_reviews(business, review)
df_t3_q4.show(truncate=False)
save_csv(df_t3_q4, results_root / "review", "q4_top_businesses_by_reviews")

### Q5 - 5-star reviews for businesses in the "Food" category
**Operations:** `filter`, `join`

In [ ]:
df_t3_q5 = t3.q5_five_star_food_reviews(business, review)
df_t3_q5.show(truncate=False)
save_csv(df_t3_q5, results_root / "review", "q5_five_star_food_reviews")

### Q6 - Cumulative review count per business over time
**Operations:** `window (count, rowsBetween)`

In [ ]:
df_t3_q6 = t3.q6_cumulative_reviews_per_business(review)
df_t3_q6.show(truncate=False)
save_csv(df_t3_q6, results_root / "review", "q6_cumulative_reviews_per_business")

## Engagement Analytics

In [ ]:
import src.analysis.transformations.engagement as t4

### Q1 - High-compliment tips written after 2020
**Operations:** `filter`

In [ ]:
df_t4_q1 = t4.q1_recent_popular_tips(tip)
df_t4_q1.show(truncate=False)
save_csv(df_t4_q1, results_root / "tip", "q1_recent_popular_tips")

### Q2 - Top 20 most checked-in businesses
**Operations:** `groupBy`, `join`

In [ ]:
df_t4_q2 = t4.q2_most_visited_businesses(checkin, business)
df_t4_q2.show(truncate=False)
save_csv(df_t4_q2, results_root / "tip", "q2_most_visited_businesses")

### Q3 - Top 3 businesses by tip count per city
**Operations:** `groupBy`, `join`, `window (rank)`

In [ ]:
df_t4_q3 = t4.q3_top_tipped_businesses_per_city(tip, business)
df_t4_q3.show(30, truncate=False)
save_csv(df_t4_q3, results_root / "tip", "q3_top_tipped_businesses_per_city")

### Q4 - Businesses with both tips and photos
**Operations:** `groupBy`, `join` (multiple)

In [ ]:
df_t4_q4 = t4.q4_businesses_with_tips_and_photos(tip, photo, business)
df_t4_q4.show(truncate=False)
save_csv(df_t4_q4, results_root / "tip", "q4_businesses_with_tips_and_photos")

### Q5 - Open, highly-rated businesses (≥ 4.0) with ≥ 10 tips
**Operations:** `filter`, `join`

In [ ]:
df_t4_q5 = t4.q5_open_quality_businesses_with_tips(tip, business)
df_t4_q5.show(truncate=False)
save_csv(df_t4_q5, results_root / "tip", "q5_open_quality_businesses_with_tips")

### Q6 - Days between consecutive tips per user
**Operations:** `window (lag)`, `datediff`

In [ ]:
df_t4_q6 = t4.q6_tip_frequency_per_user(tip)
df_t4_q6.show(truncate=False)
save_csv(df_t4_q6, results_root / "tip", "q6_tip_frequency_per_user")

---
## Done
All 24 questions executed. Results are in `results/` (gitignored).

In [ ]:
spark.stop()
print("Spark session stopped.")